# Notebook 2

Secondo tentativo svolto.
L'architettura della rete segue la stessa vista nel notebook 1 (con differenza che il numero di layer è parametrizzato), con possibile aggiunta della possibilità di inserire un secondo layer convoluzionale prima del pooling (simile a VGGNet) (che in pratica ha portato scarsi risultati forse per la dimensione del dataset).
La principale aggiunta è "l'online data augmentation" (sostanzialmente a ogni epoca le immagini vengono passate attraverso dei filtri per evitare overfitting), si differzia da offline in quanto questi immagini vengono generate a runtime

In più si è provato a svolgere una sottospecie di baggin (amalgama di 5 reti), per verificare se questo aumenta o meno la performance della rete (1-1.5%)

# Imports

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import BatchNormalization
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import GlobalAveragePooling2D
from keras.layers import Dropout
from keras.layers import ReLU
from keras.layers import Rescaling
from keras.layers import RandomFlip
from keras.layers import RandomRotation
from keras.layers import RandomZoom
from keras.optimizers import AdamW
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import keras

import tensorflow as tf

## Download dataset

In [ ]:

DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATASET_NAME = "flower_photos"
CACHE_DIR = "."

data_dir = pathlib.Path(
    tf.keras.utils.get_file(
        origin    = DATASET_URL,
        fname     = DATASET_NAME,
        cache_dir = CACHE_DIR,
        untar     = True,
        extract   = True,
    )
)

### Parameters

In [ ]:
IMG_HEIGHT = 180
IMG_WIDTH  = 180
BATCH_SIZE = 32
SEED       = 67
MAX_INTERNAL_LAYERS = 4
ADD_SECOND_CONV_LAYER = False
ADD_ONLINE_DATA_AUGMENTATION = True

TRAIN_SPLIT = 0.75
VAL_SPLIT  = 0.15
AUTOTUNE = tf.data.AUTOTUNE

Load the downloaded dataset and split it into train, validation and test sets

In [ ]:
full_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

total_batches = len(full_ds)

train_size = int(TRAIN_SPLIT * total_batches)
val_size   = int(VAL_SPLIT * total_batches)

train_ds = full_ds.take(train_size)
val_ds   = full_ds.skip(train_size).take(val_size)
test_ds  = full_ds.skip(train_size + val_size)

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Layers

In [ ]:
ensamble = []
history = []
for i in range(5):
    model = Sequential()
    model.add(Rescaling(1./255))

    if ADD_ONLINE_DATA_AUGMENTATION:
        model.add(RandomFlip("horizontal"))
        model.add(RandomRotation(0.2))
        model.add(RandomZoom(0.1))

    for j in range(5, MAX_INTERNAL_LAYERS + 5):
        model.add(Conv2D(2 ** j, 5 if j == 5 else 3, activation='relu', padding='same'))
        model.add(BatchNormalization())
        model.add(MaxPooling2D())

        if ADD_SECOND_CONV_LAYER:
            model.add(Conv2D(2 ** j, 3, padding='same'))
            model.add(BatchNormalization())
            model.add(ReLU())

        if j < MAX_INTERNAL_LAYERS + 4:
            model.add(MaxPooling2D())

    model.add(GlobalAveragePooling2D())

    model.add(Dense(2 ** (MAX_INTERNAL_LAYERS + 5 - 1), activation='relu'))
    model.add(Dropout(0.5))

    model.add(Dense(5))

    model.compile(
        optimizer=AdamW(1e-4, 1e-4),
        loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'])
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=f"models/best_checkpoint{i}.keras",
            monitor='val_accuracy',
            mode='max',
            save_best_only=True)
    ]

    history.append(model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=100,
        callbacks=callbacks
    ))

    ensamble.append(model)

In [ ]:
def build_ensemble(models):
    inp = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    
    outputs = []
    for i, model in enumerate(models):
        model.name = f"model_{i}"
        outputs.append(model(inp, training=False))
    
    avg = keras.layers.Average()(outputs)
    
    return keras.Model(inputs=inp, outputs=avg)

ensamble = [
    keras.models.load_model("backup/best_checkpoint1.keras"),
    keras.models.load_model("backup/best_checkpoint2.keras"),
    keras.models.load_model("backup/best_checkpoint3.keras"),
    keras.models.load_model("backup/best_checkpoint.keras"),
    keras.models.load_model("backup/best_checkpoint8.keras"),
]

ensemble_model = build_ensemble(ensamble)

ensemble_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

for e in ensamble:
    loss, accuracy = e.evaluate(test_ds)
    print(f"Ensemble Accuracy: {accuracy:.4f}")

loss, accuracy = ensemble_model.evaluate(test_ds)
print(f"Ensemble Accuracy: {accuracy:.4f}")

In [ ]:
N = 100
i = 0
for e in history:
    plt.style.use("ggplot")
    plt.figure()
    plt.plot(np.arange(0, N), e.history["loss"], label="train_loss")
    plt.plot(np.arange(0, N), e.history["val_loss"], label="val_loss")
    plt.plot(np.arange(0, N), e.history["accuracy"], label="train_acc")
    plt.plot(np.arange(0, N), e.history["val_accuracy"], label="val_acc")
    plt.title("Training Loss and Accuracy on Dataset")
    plt.xlabel("Epoch #")
    plt.ylabel("Loss/Accuracy")
    plt.legend(loc="lower left")
    plt.savefig(f"plot{i}.png")
    i = i + 1